# Day 1: AI-Driven Defect Prediction & Process Control in AM
### Duke University | AI in Manufacturing Systems
### Created by Dr. Jacob Peloquin, NC State University


This notebook has **two activities**:

| Activity | Focus | Key interaction |
|---|---|---|
| **A1** | Raw sensor signals → defect predictions | Tune sensor weights, read correlation results |
| **A2** | Defect predictions → engineering decisions | Set intervention threshold, analyze cost tradeoff |

**Files needed (place in the same folder as this notebook):**
- `day1_am_bins.csv` — bin-level features & defect labels
- `day1_am_temporal.csv` — raw temporal signals (O₂, Temp, Power)
- `day1_am_meltpool.csv` — melt pool spatial map per layer
- `day1_am_optical.csv` — optical image per layer


### ▶ Run cells in order — start here:

In [1]:
import sys

def _is_pyodide():
    try: import pyodide; return True
    except ImportError: return False

def _is_colab():
    try: import google.colab; return True
    except ImportError: return False

if _is_pyodide():
    print("JupyterLite: run  import micropip; await micropip.install(['seaborn','scikit-learn'])  in a new cell.")
else:
    import subprocess
    flags = [sys.executable, "-m", "pip", "install", "-q", "seaborn", "scikit-learn"]
    if not _is_colab():
        flags.append("--break-system-packages")
    subprocess.run(flags, capture_output=True)
    print("Packages ready.")


Packages ready.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SETUP — Load datasets & train models. Run once.            ║
# ╚══════════════════════════════════════════════════════════════╝

import os, urllib.request

BASE = "https://raw.githubusercontent.com/JPELO/Duke_AI4AM/main/"
files = [
    "day1_am_bins.csv",
    "day1_am_temporal.csv",
    "day1_am_meltpool.csv",
    "day1_am_optical.csv"
]

for f in files:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + f, f)

print("Data files ready.")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, seaborn as sns, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi':100,'axes.spines.top':False,'axes.spines.right':False})

# ── Load datasets ──────────────────────────────────────────────
df       = pd.read_csv('day1_am_bins.csv')
df_temp  = pd.read_csv('day1_am_temporal.csv')
df_melt  = pd.read_csv('day1_am_meltpool.csv')
df_opt   = pd.read_csv('day1_am_optical.csv')

# ── Constants (must match dataset generation) ──────────────────
N_LAYERS = int(df['layer'].max()) + 1
GRID     = int(df['bin_x'].max()) + 1
IMG_SIZE = 32
BIN_PIX  = IMG_SIZE // GRID

# ── Sensor / defect metadata ───────────────────────────────────
S_FEATS  = [['s1_mean','s1_std'], ['s2_mean','s2_std'],
            ['s3_mean','s3_std','s3_max'], ['s4_mean','s4_std'],
            ['s5_mean','s5_std']]
S_NAMES  = ['O₂ Level','Chamber Temp','Melt Pool Temp','Optical Image','Laser Power']
S_UNITS  = ['% O₂','°C','°C','Reflectance (0-1)','Watts']
DEFECTS  = ['porosity','spatter','super_elevation']
D_NAMES  = ['Porosity','Spatter','Super-elevation']
ALL_F    = [f for g in S_FEATS for f in g]

# ── Train 5 sub-models × 3 defects ────────────────────────────
sub_models = {}
for d in DEFECTS:
    sub_models[d] = {}
    y = df[d].values
    for i,feats in enumerate(S_FEATS):
        sc  = StandardScaler().fit(df[feats])
        clf = LogisticRegression(max_iter=500,class_weight='balanced',random_state=42)
        clf.fit(sc.transform(df[feats]), y)
        sub_models[d][i] = (clf, sc, feats)

# ── Helpers ────────────────────────────────────────────────────
def predict_w(weights, defect, data=df):
    w = np.array(weights,float); w /= w.sum()
    probs = np.zeros(len(data))
    for i,feats in enumerate(S_FEATS):
        clf,sc,_ = sub_models[defect][i]
        probs += w[i]*clf.predict_proba(sc.transform(data[feats]))[:,1]
    return (probs>0.5).astype(int), probs

def eval_weights(weights, data=df):
    return {dn: f1_score(data[d], predict_w(weights,d,data)[0], zero_division=0)
            for d,dn in zip(DEFECTS,D_NAMES)}

def bar_f1(results, title, ax=None):
    show = ax is None
    if show: fig,ax = plt.subplots(figsize=(6,3))
    colors = ['#e74c3c' if v<0.45 else '#f39c12' if v<0.65 else '#2ecc71'
              for v in results.values()]
    bars = ax.bar(results.keys(), results.values(), color=colors, edgecolor='white', width=0.45)
    ax.set_ylim(0,1); ax.set_ylabel('F1 Score')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axhline(0.65,color='green',lw=1,ls='--',alpha=0.5)
    ax.axhline(0.45,color='orange',lw=1,ls='--',alpha=0.5)
    for bar,v in zip(bars,results.values()):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}',
                ha='center',fontsize=9,fontweight='bold')
    if show: plt.tight_layout(); plt.show()

# ── Helper: get layer image from CSV ──────────────────────────
def get_melt_layer(layer_idx):
    sub = df_melt[df_melt['layer']==layer_idx]
    img = np.zeros((IMG_SIZE,IMG_SIZE))
    for _,row in sub.iterrows():
        img[int(row['x']),int(row['y'])] = row['temp']
    return img

def get_opt_layer(layer_idx):
    sub = df_opt[df_opt['layer']==layer_idx]
    img = np.zeros((IMG_SIZE,IMG_SIZE))
    for _,row in sub.iterrows():
        img[int(row['x']),int(row['y'])] = row['reflectance']
    return img

print(f"✅  Bins dataset:    {df.shape[0]} rows × {df.shape[1]} cols  |  {N_LAYERS} layers, {GRID}×{GRID} grid")
print(f"    Temporal:       {df_temp.shape[0]} rows  ({N_LAYERS} layers × {len(df_temp)//N_LAYERS} timesteps)")
print(f"    Melt pool map:  {df_melt.shape[0]} rows  ({N_LAYERS} layers × {IMG_SIZE}×{IMG_SIZE} px)")
print(f"    Optical image:  {df_opt.shape[0]} rows  ({N_LAYERS} layers × {IMG_SIZE}×{IMG_SIZE} px)")
print()
rates = {dn:df[d].mean() for d,dn in zip(DEFECTS,D_NAMES)}
print("Defect rates:", "  |  ".join(f"{k}: {v:.1%}" for k,v in rates.items()))
print(f"Sub-models trained: {len(DEFECTS)} defects × {len(S_FEATS)} sensors")


✅  Bins dataset:    720 rows × 17 cols  |  45 layers, 4×4 grid
    Temporal:       1800 rows  (45 layers × 40 timesteps)
    Melt pool map:  46080 rows  (45 layers × 32×32 px)
    Optical image:  46080 rows  (45 layers × 32×32 px)

Defect rates: Porosity: 51.1%  |  Spatter: 30.4%  |  Super-elevation: 27.8%
Sub-models trained: 3 defects × 5 sensors


---
# 🧪 Activity 1 — From Raw Signals to Defect Predictions

**The pipeline:**
```
5 Sensor Streams → Feature Extraction → 5 Sub-models → Weighted Ensemble → Binary Defect Prediction
```

| Defect | Root cause | Key sensors |
|---|---|---|
| **Porosity** | Energy deficit (low power) or O₂ contamination | Power ↓, O₂ ↑ |
| **Spatter** | Excess energy density / melt pool overheating | Power ↑, Melt ↑ |
| **Super-elevation** | Localized thermal hot spots | Melt max ↑, Melt std ↑ |


### Cell A1-1 · Signal Visualization

Set `SIGNAL` (1–5) to choose which sensor to visualize.  
For **Sensor 3 (Melt Pool)** and **Sensor 4 (Optical)**, also set the layer number to display.

> **Instructor note:** Layer numbers below are pre-set to recommended layers.
> **Best for Melt Pool (Signal 3):** L10, L17, L19, L20 — highest thermal std and hot-spot count
> **Best for Optical (Signal 4):** L0, L7, L9, L13, L17, L21 — clearest hatch pattern (~50% melt / ~50% powder)


In [ ]:
# ┌───────────────────────────────────────────────────────────────────┐
# │  SIGNAL: 1=O₂  2=Chamber Temp  3=Melt Pool  4=Optical  5=Power  │
# ├───────────────────────────────────────────────────────────────────┤
# │  For signals 3 & 4, set the layer to display (0 – 44)            │
# │  Best for Melt Pool:  10, 18, 25, 38  (high thermal std)          │
# │  Best for Optical:  0, 7, 12, 20, 31, 40  (~50% melt / ~50% powder) │
# └───────────────────────────────────────────────────────────────────┘
SIGNAL       = 3
LAYER_MELT   = 10    # layer shown in melt pool heatmap  (signal 3)
LAYER_OPT    =  0    # layer shown in optical image       (signal 4)

# ── plot ────────────────────────────────────────────────────────────
_info = {1:('O₂ Level','s1_oxy','% O₂','temporal'),
         2:('Chamber Temp','s2_ctemp','°C','temporal'),
         3:('Melt Pool Temp',None,'°C','spatiotemporal'),
         4:('Post-Melt Optical',None,'Reflectance','image'),
         5:('Laser Power','s5_power','Watts','temporal')}
name,col,ylabel,kind = _info[SIGNAL]

if kind == 'temporal':
    fig,ax = plt.subplots(figsize=(13,3.2))
    ax.plot(df_temp['t_abs'], df_temp[col], lw=0.8, color='steelblue')
    for L in range(N_LAYERS):
        ax.axvline(L*1.05, color='gray', lw=0.3, alpha=0.5)
    ax.set_xlabel('Time — layers separated by gaps (a.u.)')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Signal {SIGNAL}: {name}  |  All {N_LAYERS} layers', fontweight='bold')
    plt.tight_layout(); plt.show()

elif kind == 'spatiotemporal':
    # Left: mean per layer; Right: chosen layer heatmap
    layer_means = df_melt.groupby('layer')['temp'].mean().values
    fig,(a1,a2) = plt.subplots(1,2,figsize=(13,4.5))
    a1.plot(range(N_LAYERS), layer_means, 'o-', color='tomato', ms=5, lw=1.8)
    a1.axvline(LAYER_MELT, color='steelblue', lw=1.5, ls='--', label=f'Displayed layer (L{LAYER_MELT})')
    a1.set_xlabel('Layer'); a1.set_ylabel('Mean Melt-Pool Temp (°C)')
    a1.set_title('Mean melt-pool temperature per layer', fontweight='bold')
    a1.legend(fontsize=9)
    melt_img = get_melt_layer(LAYER_MELT)
    im = a2.imshow(melt_img, cmap='inferno', origin='upper')
    plt.colorbar(im, ax=a2, label='Temp (°C)')
    for g in range(1,GRID):
        a2.axhline(g*BIN_PIX-0.5, color='cyan', lw=0.9, alpha=0.85)
        a2.axvline(g*BIN_PIX-0.5, color='cyan', lw=0.9, alpha=0.85)
    a2.set_title(f'Melt-pool spatial map — Layer {LAYER_MELT}  (cyan = bin grid)', fontweight='bold')
    a2.set_xlabel('X (px)'); a2.set_ylabel('Y (px)')
    # Annotate hot spots
    hot_thresh = melt_img.mean() + 2*melt_img.std()
    hot_coords = np.argwhere(melt_img > hot_thresh)
    if len(hot_coords):
        a2.scatter(hot_coords[:,1], hot_coords[:,0], s=6, c='white', alpha=0.5, label='Hot spot')
        a2.legend(fontsize=8, loc='lower right')
    plt.tight_layout(); plt.show()
    print(f"Layer {LAYER_MELT} — Mean: {melt_img.mean():.1f} °C  |  "
          f"Max: {melt_img.max():.1f} °C  |  Std: {melt_img.std():.1f}  |  "
          f"Hot-spot pixels (>mean+2σ): {len(hot_coords)}")

elif kind == 'image':
    opt_img = get_opt_layer(LAYER_OPT)
    fig,ax = plt.subplots(figsize=(5.5,5.5))
    im = ax.imshow(opt_img, cmap='gray', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='Reflectance (0–1)', fraction=0.046)
    for g in range(1,GRID):
        ax.axhline(g*BIN_PIX-0.5, color='lime', lw=1.2)
        ax.axvline(g*BIN_PIX-0.5, color='lime', lw=1.2)
    ax.set_title(f'Signal {SIGNAL}: {name} — Layer {LAYER_OPT}  (green = bins)', fontweight='bold')
    ax.set_xlabel('X (px)'); ax.set_ylabel('Y (px)')
    plt.tight_layout(); plt.show()
    print(f"Layer {LAYER_OPT} — Mean reflectance: {opt_img.mean():.4f}  |  Std: {opt_img.std():.4f}")


### Cell A1-2 · Defect Type Examples — synthetic optical appearance

In [ ]:
rng2 = np.random.default_rng(99)
fig, axes = plt.subplots(1,3,figsize=(12,4))

# Porosity — dark circular voids scattered across layer
c1 = rng2.normal(0.55,0.06,(64,64))
for _ in range(14):
    cx,cy = rng2.integers(4,60,2); r=rng2.integers(2,5)
    yy,xx = np.ogrid[:64,:64]
    c1[(xx-cx)**2+(yy-cy)**2 < r**2] *= rng2.uniform(0.2,0.45)
axes[0].imshow(np.clip(c1,0,1),cmap='gray',vmin=0,vmax=1)
axes[0].set_title('Porosity - Dark subsurface voids', fontweight='bold'); axes[0].axis('off')

# Spatter — bright ejected particles
c2 = rng2.normal(0.45,0.05,(64,64))
for cx,cy in rng2.integers(0,64,(25,2)):
    c2[cx,cy] = rng2.uniform(0.82,1.0)
    for dx,dy in [(-1,0),(1,0),(0,-1),(0,1)]:
        nx,ny=cx+dx,cy+dy
        if 0<=nx<64 and 0<=ny<64: c2[nx,ny] = rng2.uniform(0.65,0.85)
axes[1].imshow(np.clip(c2,0,1),cmap='gray',vmin=0,vmax=1)
axes[1].set_title('Spatter - Bright ejected particles', fontweight='bold'); axes[1].axis('off')

# Super-elevation — bright raised ridge
c3 = rng2.normal(0.45,0.04,(64,64))
ridge_y = (32+10*np.sin(np.linspace(0,2*np.pi,64))).astype(int)
for xi,yi in enumerate(ridge_y):
    for dyi in range(-2,3):
        yy2 = np.clip(yi+dyi,0,63)
        c3[yy2,xi] = np.clip(0.80-0.08*abs(dyi)+rng2.normal(0,0.04),0,1)
axes[2].imshow(np.clip(c3,0,1),cmap='gray',vmin=0,vmax=1)
axes[2].set_title('Super-elevation - Bright raised ridge', fontweight='bold'); axes[2].axis('off')

plt.suptitle('Defect Types — Post-Melt Optical Image Appearance (Synthetic)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print("Ground truth labels assigned via ex-situ microCT. These images show what each defect looks like.")


### Cell A1-3 · Statistical Summary

In [ ]:
display_cols = ['s1_mean','s2_mean','s3_mean','s3_max','s4_mean','s5_mean',
                'porosity','spatter','super_elevation']
print("── Feature statistics ──")
display(df[display_cols].describe().round(3))
print()
print("── Defect prevalence ──")
for d,dn in zip(DEFECTS,D_NAMES):
    n = df[d].sum()
    print(f"  {dn:<18}: {n:>3} / {len(df)} bins  ({df[d].mean():.1%})")


### Cell A1-4 · Column Definitions

In [ ]:
col_defs = {
  'layer':           'Layer index (0 – 24)',
  'bin_x / bin_y':   'Spatial bin index (0–3) in the 4×4 grid',
  's1_mean/std':     'Sensor 1 — Chamber O₂ (%): mean & std in this bin time-window',
  's2_mean/std':     'Sensor 2 — Chamber Temperature (°C): mean & std',
  's3_mean/std/max': 'Sensor 3 — Melt Pool Temp (°C): mean, std, max in this bin footprint',
  's4_mean/std':     'Sensor 4 — Optical Image reflectance (0–1): mean & std in bin crop',
  's5_mean/std':     'Sensor 5 — Laser Power (W): mean & std in this bin time-window',
  'porosity':        'Ground truth — 1 = porosity detected (ex-situ microCT)',
  'spatter':         'Ground truth — 1 = spatter detected',
  'super_elevation': 'Ground truth — 1 = super-elevation detected',
}
print(f"{'Column':<24}  Description")
print("─"*76)
for k,v in col_defs.items(): print(f"  {k:<22}  {v}")


### Cell A1-5 · First Weight Attempt — *random guess*

Each weight controls how much that sensor's sub-model contributes to the final prediction.  
Weights are auto-normalized so they don't need to sum exactly to 1.  
Re-run after changing values to update the F1 scores.


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  Adjust these weights and re-run  (will be normalized to 1.0)  │
# └─────────────────────────────────────────────────────────────────┘
W_OXYGEN     = 0.20   # Sensor 1 · Chamber O₂
W_CTEMP      = 0.20   # Sensor 2 · Chamber Temperature
W_MELTPOOL   = 0.20   # Sensor 3 · Melt Pool Temperature
W_OPTICAL    = 0.20   # Sensor 4 · Post-Melt Optical Image
W_LASERPOWER = 0.20   # Sensor 5 · Laser Power

weights_v1 = [W_OXYGEN, W_CTEMP, W_MELTPOOL, W_OPTICAL, W_LASERPOWER]
w1 = np.array(weights_v1, float); w1 /= w1.sum()
results_v1 = eval_weights(w1)
bar_f1(results_v1,
       f'Attempt 1  |  O₂:{w1[0]:.2f}  Temp:{w1[1]:.2f}  '
       f'Melt:{w1[2]:.2f}  Opt:{w1[3]:.2f}  Pwr:{w1[4]:.2f}')
print("F1 scores:", {k:f'{v:.3f}' for k,v in results_v1.items()})
print("(Green ≥ 0.65 = good  |  Orange 0.45–0.65 = marginal  |  Red < 0.45 = poor)")


### Cell A1-6 · Signal Correlation Matrix
Which sensor features correlate most strongly with which defects?  
Use this to **inform your second weight attempt below**.


In [ ]:
# ┌───────────────────────────────────────────────────────────────────┐
# │  Set DEFECT_FOCUS to zoom in on one defect (bar chart view)      │
# │  'porosity' | 'spatter' | 'super_elevation' | None (full matrix) │
# └───────────────────────────────────────────────────────────────────┘
DEFECT_FOCUS = None

corr_sub = df[ALL_F + DEFECTS].corr().loc[ALL_F, DEFECTS]
corr_sub.columns = D_NAMES
boundaries = np.cumsum([0]+[len(g) for g in S_FEATS])

if DEFECT_FOCUS:
    d_idx = DEFECTS.index(DEFECT_FOCUS)
    vals  = corr_sub.iloc[:, d_idx]
    fig,ax = plt.subplots(figsize=(8,3.8))
    colors = ['#c0392b' if v>0 else '#2980b9' for v in vals]
    ax.barh(ALL_F, vals, color=colors, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Pearson Correlation with ' + D_NAMES[d_idx])
    ax.set_title(f'Feature correlations → {D_NAMES[d_idx]}', fontweight='bold')
    for b0,b1,sn in zip(boundaries[:-1],boundaries[1:],S_NAMES):
        ax.axhline(b1-0.5, color='gray', lw=0.5, alpha=0.5)
        ax.text(ax.get_xlim()[0]-0.005,(b0+b1)/2, sn[:6],
                ha='right', va='center', fontsize=7.5, color='gray')
    plt.tight_layout(); plt.show()
else:
    fig,ax = plt.subplots(figsize=(7,7))
    sns.heatmap(corr_sub, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                linewidths=0.5, ax=ax, vmin=-0.7, vmax=0.7, square=True)
    for b0,b1,sn in zip(boundaries[:-1],boundaries[1:],S_NAMES):
        ax.axhline(b1, color='white', lw=2)
        ax.text(-0.14,(b0+b1)/2/len(ALL_F), f'S{S_NAMES.index(sn)+1}',
                transform=ax.transAxes, ha='right', va='center',
                fontsize=9, fontweight='bold')
    ax.set_title('Sensor Feature → Defect Correlation', fontweight='bold')
    plt.tight_layout(); plt.show()
print("Red/warm = positive correlation (feature ↑ → defect more likely)")
print("Blue/cool = negative correlation (feature ↑ → defect less likely)")


### Cell A1-7 · Sensor Usefulness — Individual Sub-Model Performance
How well can each sensor **on its own** predict each defect?  
Each bar shows the F1 score of a logistic regression trained on **only that sensor's features**.  
Tall bar = that sensor is independently useful for that defect.  
Use this alongside the correlation matrix to inform your second weight attempt.


In [ ]:
# ┌────────────────────────────────────────────────────────────────────┐
# │  Set DEFECT_TYPE and re-run — or view all three at once           │
# │  'porosity' | 'spatter' | 'super_elevation' | 'all'              │
# └────────────────────────────────────────────────────────────────────┘
DEFECT_TYPE = 'all'

from sklearn.model_selection import cross_val_score

def sensor_f1_scores(defect):
    y = df[defect].values
    scores = []
    for feats in S_FEATS:
        sc  = StandardScaler()
        X   = sc.fit_transform(df[feats])
        clf = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
        cv  = cross_val_score(clf, X, y, cv=4, scoring='f1').mean()
        scores.append(cv)
    return scores

if DEFECT_TYPE == 'all':
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
    for ax, defect, dname in zip(axes, DEFECTS, D_NAMES):
        scores = sensor_f1_scores(defect)
        colors = ['#2ecc71' if s==max(scores) else
                  '#f39c12' if s > 0.60 else '#e74c3c' for s in scores]
        bars = ax.bar(S_NAMES, scores, color=colors, edgecolor='white', width=0.55)
        ax.set_ylim(0, 1); ax.set_ylabel('F1 Score (4-fold CV)')
        ax.set_title(f'{dname}', fontweight='bold', fontsize=11)
        ax.axhline(0.60, color='gray', lw=1, ls='--', alpha=0.5)
        ax.set_xticklabels(S_NAMES, rotation=18, ha='right', fontsize=9)
        for bar, v in zip(bars, scores):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}',
                    ha='center', fontsize=8.5, fontweight='bold')
    plt.suptitle('Sensor Usefulness — Individual Sub-Model F1 per Defect Type',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()
    print("Green = highest-performing sensor for that defect")
    print("Which sensors dominate each defect? Use this to inform your weights below.")
else:
    scores = sensor_f1_scores(DEFECT_TYPE)
    colors = ['#2ecc71' if s==max(scores) else
              '#f39c12' if s > 0.60 else '#e74c3c' for s in scores]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.bar(S_NAMES, scores, color=colors, edgecolor='white', width=0.5)
    ax.set_ylim(0,1); ax.set_ylabel('F1 Score (4-fold CV)')
    ax.set_title(f'Sensor Usefulness — {DEFECT_TYPE.replace("_"," ").title()}',
                 fontweight='bold')
    ax.axhline(0.60, color='gray', lw=1, ls='--', alpha=0.5, label='F1 = 0.60 reference')
    ax.set_xticklabels(S_NAMES, rotation=15, ha='right')
    ax.legend(fontsize=8)
    for bar, v in zip(bars, scores):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}',
                ha='center', fontsize=9, fontweight='bold')
    plt.tight_layout(); plt.show()
    best = S_NAMES[scores.index(max(scores))]
    print(f"Most useful sensor for {DEFECT_TYPE}: {best} (F1={max(scores):.3f})")
    print("Try DEFECT_TYPE = 'all' to compare all three defects at once.")


### Cell A1-8 · Second Weight Attempt — *use your analysis*

Look at the correlation matrix (A1-6) and model reliance (A1-7).  
Adjust weights to reflect which sensors matter most for each defect.  
The chart compares your first attempt (equal weights) vs. this informed attempt.


In [ ]:
# ┌──────────────────────────────────────────────────────────────────┐
# │  Update weights based on correlation + reliance analysis        │
# └──────────────────────────────────────────────────────────────────┘
W_OXYGEN     = 0.15   # Sensor 1 · O₂
W_CTEMP      = 0.05   # Sensor 2 · Chamber Temp
W_MELTPOOL   = 0.40   # Sensor 3 · Melt Pool  ← likely high importance
W_OPTICAL    = 0.15   # Sensor 4 · Optical
W_LASERPOWER = 0.25   # Sensor 5 · Laser Power

weights_v2 = [W_OXYGEN, W_CTEMP, W_MELTPOOL, W_OPTICAL, W_LASERPOWER]
w2 = np.array(weights_v2, float); w2 /= w2.sum()
results_v2 = eval_weights(w2)

fig,ax = plt.subplots(figsize=(8,3.8))
x = np.arange(len(D_NAMES)); width = 0.35
ax.bar(x-width/2,[results_v1[d] for d in D_NAMES],width,
       label='Attempt 1 (equal)', color='lightgray', edgecolor='gray')
ax.bar(x+width/2,[results_v2[d] for d in D_NAMES],width,
       label='Attempt 2 (informed)', color='steelblue', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(D_NAMES)
ax.set_ylim(0,1); ax.set_ylabel('F1 Score')
ax.set_title('Equal vs Informed Weights — did your analysis improve predictions?', fontweight='bold')
ax.axhline(0.65,color='green',lw=1,ls='--',alpha=0.5,label='Good (F1 ≥ 0.65)')
ax.legend(fontsize=8)
for xi,(v1,v2) in enumerate(zip([results_v1[d] for d in D_NAMES],
                                 [results_v2[d] for d in D_NAMES])):
    delta = v2-v1
    ax.text(xi+width/2, v2+0.025, f'{delta:+.2f}', ha='center', fontsize=8.5,
            fontweight='bold', color='#27ae60' if delta>0 else '#e74c3c')
plt.tight_layout(); plt.show()

OPTIMAL_WEIGHTS = w2.tolist()
print(f"✅ Optimal weights saved for Activity 2: {[round(w,3) for w in OPTIMAL_WEIGHTS]}")


---
# 🏗️ Activity 2 — Defect Predictions → Engineering Decisions

**Scenario:** You are monitoring **6 live builds** (Parts A–F) with different process conditions.  
Using the weights from Activity 1, the pipeline predicts bin-level defects layer by layer.  
A second aggregation model converts these into a **layer-level defect probability**.  
Your job: set an intervention threshold and decide what to do with each part.


### Cell A2-1 · Generate 6-Part Build Data — *run once*

In [ ]:
PART_PROFILES = {
    'A': dict(power_bias=  0, oxy_bias=0.00, label='Nominal build'),
    'B': dict(power_bias=-30, oxy_bias=0.06, label='O₂ contamination + low power'),
    'C': dict(power_bias=+35, oxy_bias=0.00, label='Excessive laser power'),
    'D': dict(power_bias=-15, oxy_bias=0.03, label='Mild mixed issues'),
    'E': dict(power_bias=+20, oxy_bias=0.00, label='Moderate power excess'),
    'F': dict(power_bias=-35, oxy_bias=0.08, label='Severe: low power + high O₂'),
}

T_PER_L2 = 40

def gen_part(profile, seed):
    np.random.seed(seed)
    pb,ob = profile['power_bias'], profile['oxy_bias']
    rows_p = []
    for L in range(N_LAYERS):
        t  = np.linspace(0,1,T_PER_L2)
        pw = np.random.normal(200+pb,18,T_PER_L2) + 8*np.sin(2*np.pi*t*4)
        ow = np.random.beta(2,20,T_PER_L2)*0.13 + 0.025 + ob
        cw = np.random.normal(85,8,T_PER_L2)
        melt_base = 1750+3.5*(200+pb-200)
        mm  = np.random.normal(melt_base,80,(IMG_SIZE,IMG_SIZE))
        for _ in range(np.random.randint(0,4)):
            cx,cy = np.random.randint(4,28,2)
            mm[cx-3:cx+3,cy-3:cy+3] += np.random.uniform(120,350)
        oo = np.clip(np.random.normal(0.5,0.08,(IMG_SIZE,IMG_SIZE))
                     + 0.08*(mm-mm.mean())/max(mm.std(),1),0,1)
        for bx in range(GRID):
            for by in range(GRID):
                ts,te = int(bx*T_PER_L2/GRID),int((bx+1)*T_PER_L2/GRID)
                p0,p1 = bx*BIN_PIX,(bx+1)*BIN_PIX
                q0,q1 = by*BIN_PIX,(by+1)*BIN_PIX
                ow_b=ow[ts:te]; cw_b=cw[ts:te]; pw_b=pw[ts:te]
                mw=mm[p0:p1,q0:q1]; iw=oo[p0:p1,q0:q1]
                rows_p.append(dict(
                    part=list(PART_PROFILES.keys())[list(PART_PROFILES.values()).index(profile)],
                    layer=L, bin_x=bx, bin_y=by,
                    s1_mean=ow_b.mean(),s1_std=ow_b.std(),
                    s2_mean=cw_b.mean(),s2_std=cw_b.std(),
                    s3_mean=mw.mean(),s3_std=mw.std(),s3_max=mw.max(),
                    s4_mean=iw.mean(),s4_std=iw.std(),
                    s5_mean=pw_b.mean(),s5_std=pw_b.std()))
    return pd.DataFrame(rows_p)

try:
    OPT_W = OPTIMAL_WEIGHTS
    print(f"Using weights from Activity 1: {[round(w,3) for w in OPT_W]}")
except NameError:
    OPT_W = [0.15, 0.05, 0.40, 0.15, 0.25]
    print(f"Activity 1 skipped — using default weights: {OPT_W}")

parts_dfs = {}
for pid,pname in enumerate(PART_PROFILES):
    PART_PROFILES[pname]['_id'] = pname
    pdf = gen_part(PART_PROFILES[pname], seed=100+pid)
    for d in DEFECTS:
        pdf[f'pred_{d}'], pdf[f'prob_{d}'] = predict_w(OPT_W, d, data=pdf)
    parts_dfs[pname] = pdf

print(f"✅ 6-part dataset ready  ({sum(len(v) for v in parts_dfs.values())} total bins)")
for pname,prof in PART_PROFILES.items():
    pdf = parts_dfs[pname]
    rates = "  ".join(f"{d[:3].capitalize()}:{pdf[f'pred_{d}'].mean():.0%}" for d in DEFECTS)
    print(f"  Part {pname}: {prof['label']:<38}  {rates}")


### Cell A2-2 · Defect Prediction Map — set `PART` and re-run

In [ ]:
# ┌──────────────────────────────────────────────────────────┐
# │  PART = 'A' | 'B' | 'C' | 'D' | 'E' | 'F'             │
# └──────────────────────────────────────────────────────────┘
PART = 'B'

pdf   = parts_dfs[PART]
label = PART_PROFILES[PART]['label']

fig,axes = plt.subplots(1,3,figsize=(13,4.5))
for ax,defect,dname in zip(axes,DEFECTS,D_NAMES):
    grid = pdf.pivot_table(index='layer',columns='bin_x',
                           values=f'pred_{defect}',aggfunc='mean')
    im = ax.imshow(grid.values, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'{dname} - Fraction of bins flagged', fontsize=9, fontweight='bold')
    ax.set_xlabel('Bin X'); ax.set_ylabel('Layer')
    plt.colorbar(im, ax=ax, label='Flagged fraction')

fig.suptitle(f'Part {PART} — {label}', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Part {PART} ({label})")
for d,dn in zip(DEFECTS,D_NAMES):
    rate = pdf[f'pred_{d}'].mean()
    bar  = '█'*int(rate*30) + '░'*int((1-rate)*30)
    print(f"  {dn:<18} {bar}  {rate:.1%}")


### Cell A2-3 · Defect Probability & Intervention Threshold

A second model aggregates bin-level predictions → **layer-level defect probability**  
using spatial clustering (adjacent flagged bins) + temporal smoothing across layers.  
Set `THRESHOLD` and re-run to see which layers would trigger an intervention.


In [ ]:
def build_defect_prob(pdf, alpha=0.35):
    layers = sorted(pdf['layer'].unique())
    smoothed = []
    prev = None
    for L in layers:
        sub = pdf[pdf['layer']==L]
        any_def = (sub[[f'pred_{d}' for d in DEFECTS]].sum(axis=1)>0).values.reshape(GRID,GRID)
        raw_score = any_def.mean()
        adj = sum(0.04 for bx in range(GRID-1) for by in range(GRID-1)
                  if (any_def[bx,by] and any_def[bx+1,by]) or
                     (any_def[bx,by] and any_def[bx,by+1]))
        score = min(1.0, raw_score+adj)
        prev  = score if prev is None else alpha*score+(1-alpha)*prev
        smoothed.append(prev)
    return np.array(layers), np.array(smoothed)

part_probs = {pname: build_defect_prob(pdf) for pname,pdf in parts_dfs.items()}

# ┌─────────────────────────────────────────────────────────────┐
# │  Set THRESHOLD (0.0 – 1.0) and re-run                      │
# └─────────────────────────────────────────────────────────────┘
THRESHOLD = 0.40

fig,axes = plt.subplots(2,3,figsize=(14,7),sharey=True)
axes = axes.flatten()
for i,(pname,prof) in enumerate(PART_PROFILES.items()):
    ax = axes[i]
    layers,probs = part_probs[pname]
    ax.plot(layers,probs,color=plt.cm.tab10.colors[i],lw=2)
    ax.axhline(THRESHOLD,color='red',ls='--',lw=1.5,label=f'Threshold {THRESHOLD}')
    flagged = layers[probs>=THRESHOLD]
    for fl in flagged: ax.axvspan(fl-0.5,fl+0.5,alpha=0.2,color='red')
    ax.set_title(f'Part {pname}: {prof["label"]}', fontsize=8.5, fontweight='bold')
    ax.set_xlabel('Layer'); ax.set_ylabel('Defect Prob.')
    ax.set_ylim(0,1)
    n = len(flagged)
    ax.text(0.97,0.95,f'{n} flags',transform=ax.transAxes,ha='right',va='top',
            fontsize=8.5,color='red' if n>3 else 'orange' if n>0 else 'green',
            fontweight='bold')
    if i==0: ax.legend(fontsize=7,loc='upper left')

plt.suptitle(f'Defect Probability — All 6 Parts  |  Threshold = {THRESHOLD}',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


### Cell A2-4 · Cost Tradeoff — what does your threshold cost?

In [ ]:
# ┌──────────────────────────────────────────────────────────────────┐
# │  Adjust costs and threshold, re-run to see the tradeoff         │
# └──────────────────────────────────────────────────────────────────┘
COST_FALSE_ALARM   =  400    # $ per unnecessary intervention
COST_MISSED_DEFECT = 6000    # $ per undetected defect reaching post-process

TRUE_DEF_THRESH = 0.55  # proxy ground truth: layers above this = "true defect zone"

thresholds = np.linspace(0.05, 0.95, 200)
fp_costs,fn_costs,total_costs = [],[],[]
for t in thresholds:
    fp=fn=0
    for _,(layers,probs) in part_probs.items():
        true_def = (probs>=TRUE_DEF_THRESH).astype(int)
        pred_def = (probs>=t).astype(int)
        fp += ((pred_def==1)&(true_def==0)).sum()
        fn += ((pred_def==0)&(true_def==1)).sum()
    fp_costs.append(fp*COST_FALSE_ALARM)
    fn_costs.append(fn*COST_MISSED_DEFECT)
    total_costs.append(fp*COST_FALSE_ALARM+fn*COST_MISSED_DEFECT)

best_t = thresholds[np.argmin(total_costs)]

fig,ax = plt.subplots(figsize=(10,4.5))
ax.plot(thresholds,fp_costs,  color='steelblue',lw=1.8,label='Cost of false alarms')
ax.plot(thresholds,fn_costs,  color='tomato',   lw=1.8,label='Cost of missed defects')
ax.plot(thresholds,total_costs,color='black',   lw=2.2,label='Total cost')
ax.axvline(best_t,  color='green', ls='--',lw=1.5,label=f'Optimal ≈ {best_t:.2f}')
ax.axvline(THRESHOLD,color='orange',ls=':',lw=1.8,label=f'Your threshold = {THRESHOLD}')
ax.set_xlabel('Intervention Threshold',fontsize=11)
ax.set_ylabel('Estimated Cost ($)',fontsize=11)
ax.set_title('Cost vs Threshold  |  Adjust costs & threshold above to explore', fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

idx = np.argmin(np.abs(thresholds-THRESHOLD))
print(f"At your threshold = {THRESHOLD}:")
print(f"  False alarm cost:    ${fp_costs[idx]:>8,.0f}")
print(f"  Missed defect cost:  ${fn_costs[idx]:>8,.0f}")
print(f"  Total cost:          ${total_costs[idx]:>8,.0f}")
print(f"Cost-minimizing threshold: {best_t:.2f}  (total = ${min(total_costs):,.0f})")


---
## 📝 Team Debrief — The Five-Question Framework

| Question | Your answer |
|---|---|
| **What is the decision?** | *(when to intervene in the build)* |
| **What data inform it?** | *(which sensor streams mattered most?)* |
| **What is the model predicting?** | *(bin-level defect type → layer probability)* |
| **What constraints must hold?** | *(cost limits, part spec, qualification)* |
| **What evidence is needed to trust it?** | *(what before using this in production?)* |

**Reflection questions:**
1. Which sensor was most important — did the analysis match your intuition?
2. Was the cost-optimal threshold the same for all 6 parts? Should it be?
3. What would change if the build cost was $200,000 instead of $20,000?
4. What does a human still need to decide that this pipeline cannot?
